In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("RMIMS V2 preprocessing environment ready.")

RMIMS V2 preprocessing environment ready.


In [3]:
from pathlib import Path

print("Current notebook folder:")
print(Path.cwd())

print("\nFiles available here:")
for item in Path.cwd().iterdir():
    print(item.name)

Current notebook folder:
C:\Users\Zeanna\anaconda_projects\b33c9bb6-b656-4486-b51f-58f66e294016

Files available here:
.ipynb_checkpoints
RMIMS_FINAL_TRAINING_DATA.csv
RMIMS_RAW_DATASET_2025_TO_AUG09_2026_ACCURATE_DATES.xlsx
RMIMS_V2_Data_Preprocessing.ipynb


In [4]:
excel_file = Path.cwd() / "RMIMS_RAW_DATASET_2025_TO_AUG09_2026_ACCURATE_DATES.xlsx"

df = pd.read_excel(
    excel_file,
    sheet_name="RAW_DATA"
)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", list(df.columns))

df.head()

Dataset loaded successfully.
Rows: 17580
Columns: ['material_id', 'date_consumed', 'raw_material_name', 'unit', 'quantity_consumed']


,material_id,date_consumed,raw_material_name,unit,quantity_consumed
0,RM001,2025-01-01,Chiton,kg,16.91
1,RM002,2025-01-01,Salt,kg,4.30
2,RM003,2025-01-01,Ground Pepper,kg,1.13
3,RM004,2025-01-01,Crushed Garlic,kg,2.41
4,RM005,2025-01-01,Optional Spices or Flavorings,kg,0.93


In [5]:
# Preserve the original loaded dataset
raw_df = df.copy(deep=True)

print("Original raw dataset preserved.")
print("Original rows:", len(raw_df))
print("Original columns:", list(raw_df.columns))

Original raw dataset preserved.
Original rows: 17580
Original columns: ['material_id', 'date_consumed', 'raw_material_name', 'unit', 'quantity_consumed']


In [6]:
# Inspect the original dataset structure and data types

print("=== DATA TYPES ===")
print(raw_df.dtypes)

print("\n=== MISSING VALUES ===")
print(raw_df.isna().sum())

print("\n=== DUPLICATE ROWS ===")
print(raw_df.duplicated().sum())

print("\n=== BASIC STATISTICS ===")
print(raw_df.describe(include="all"))

=== DATA TYPES ===
material_id                  object
date_consumed        datetime64[ns]
raw_material_name            object
unit                         object
quantity_consumed           float64
dtype: object

=== MISSING VALUES ===
material_id          0
date_consumed        0
raw_material_name    0
unit                 0
quantity_consumed    0
dtype: int64

=== DUPLICATE ROWS ===
0

=== BASIC STATISTICS ===
       material_id        date_consumed raw_material_name   unit  \
count        17580                17580             17580  17580   
unique          30                  NaN                30      3   
top          RM001                  NaN            Chiton     kg   
freq           586                  NaN               586  12892   
mean           NaN  2025-10-20 12:00:00               NaN    NaN   
min            NaN  2025-01-01 00:00:00               NaN    NaN   
25%            NaN  2025-05-27 00:00:00               NaN    NaN   
50%            NaN  2025-10-20 12:00:00

In [7]:
# Approved raw materials from the RMIMS paper

paper_materials = [
    "Chiton",
    "Salt",
    "Ground Pepper",
    "Crushed Garlic",
    "Optional Spices or Flavorings",
    "Cooking Oil",
    "Small Shrimp",
    "Garlic",
    "Onion",
    "Spring Onion",
    "Cabbage",
    "Carrots",
    "Bell Pepper",
    "Soy Sauce",
    "Sesame Oil",
    "Oyster Sauce",
    "Chicken",
    "Pork",
    "Loaf Bread",
    "Butter or Margarine",
    "Sugar",
    "Pork Skin",
    "Raw Bananas",
    "Turmeric Powder",
    "Water",
    "White Sugar",
    "Peanuts",
    "Sea Salt",
    "Honey",
    "Oil"
]

source_materials = set(
    raw_df["raw_material_name"].dropna().str.strip()
)

paper_material_set = set(paper_materials)

missing_materials = sorted(
    paper_material_set - source_materials
)

extra_materials = sorted(
    source_materials - paper_material_set
)

print("Paper materials:", len(paper_materials))
print("Materials in dataset:", len(source_materials))

print("\nMISSING FROM DATASET:")
print(missing_materials if missing_materials else "NONE")

print("\nNOT IN PAPER LIST:")
print(extra_materials if extra_materials else "NONE")

Paper materials: 30
Materials in dataset: 30

MISSING FROM DATASET:
NONE

NOT IN PAPER LIST:
NONE


In [8]:
# Check which units are recorded for each raw material

unit_check = (
    raw_df
    .groupby("raw_material_name")["unit"]
    .unique()
    .reset_index()
)

unit_check.columns = [
    "raw_material_name",
    "units_found"
]

print(unit_check.to_string(index=False))

            raw_material_name units_found
                  Bell Pepper        [kg]
          Butter or Margarine        [kg]
                      Cabbage        [kg]
                      Carrots        [kg]
                      Chicken        [kg]
                       Chiton        [kg]
                  Cooking Oil         [L]
               Crushed Garlic        [kg]
                       Garlic        [kg]
                Ground Pepper        [kg]
                        Honey         [L]
                   Loaf Bread       [pcs]
                          Oil         [L]
                        Onion        [kg]
Optional Spices or Flavorings        [kg]
                 Oyster Sauce         [L]
                      Peanuts        [kg]
                         Pork        [kg]
                    Pork Skin        [kg]
                  Raw Bananas        [kg]
                         Salt        [kg]
                     Sea Salt        [kg]
                   Sesame Oil     

In [9]:
# Identify materials that have more than one recorded unit

unit_inconsistencies = (
    raw_df.groupby("raw_material_name")["unit"]
    .nunique()
    .reset_index(name="unit_count")
)

unit_inconsistencies = unit_inconsistencies[
    unit_inconsistencies["unit_count"] > 1
]

if unit_inconsistencies.empty:
    print("No unit inconsistencies found.")
else:
    print("Materials with multiple recorded units:")
    print(unit_inconsistencies.to_string(index=False))

    print("\nDetailed unit values:")
    for material in unit_inconsistencies["raw_material_name"]:
        units = (
            raw_df.loc[
                raw_df["raw_material_name"] == material,
                "unit"
            ]
            .dropna()
            .unique()
        )
        print(f"{material}: {list(units)}")

No unit inconsistencies found.


In [10]:
# Validate consumption quantities

print("=== QUANTITY VALIDATION ===")

print("Missing quantities:",
      raw_df["quantity_consumed"].isna().sum())

print("Negative quantities:",
      (raw_df["quantity_consumed"] < 0).sum())

print("Zero quantities:",
      (raw_df["quantity_consumed"] == 0).sum())

print("Positive quantities:",
      (raw_df["quantity_consumed"] > 0).sum())

print("\nMinimum quantity:",
      raw_df["quantity_consumed"].min())

print("Maximum quantity:",
      raw_df["quantity_consumed"].max())

print("Mean quantity:",
      round(raw_df["quantity_consumed"].mean(), 2))

print("Median quantity:",
      round(raw_df["quantity_consumed"].median(), 2))

=== QUANTITY VALIDATION ===
Missing quantities: 0
Negative quantities: 0
Zero quantities: 0
Positive quantities: 17580

Minimum quantity: 0.51
Maximum quantity: 53.0
Mean quantity: 7.85
Median quantity: 5.62


In [11]:
# Analyze consumption quantities separately for each raw material

material_quantity_summary = (
    raw_df
    .groupby(
        ["material_id", "raw_material_name", "unit"]
    )["quantity_consumed"]
    .agg(
        records="count",
        minimum="min",
        maximum="max",
        mean="mean",
        median="median",
        std="std"
    )
    .reset_index()
)

material_quantity_summary


,material_id,raw_material_name,unit,records,minimum,maximum,mean,median,std
0,RM001,Chiton,kg,586,12.66,26.28,18.738959,18.680,2.424667
1,RM002,Salt,kg,586,3.13,7.76,5.233720,5.210,0.702705
2,RM003,Ground Pepper,kg,586,0.82,1.82,1.254556,1.250,0.166248
3,RM004,Crushed Garlic,kg,586,1.71,3.80,2.613294,2.600,0.341760
4,RM005,Optional Spices or Flavorings,kg,586,0.51,1.13,0.835461,0.830,0.108921
5,RM006,Cooking Oil,L,586,8.29,18.65,12.541945,12.435,1.651396
6,RM007,Small Shrimp,kg,586,4.76,10.02,7.293379,7.275,0.914649
7,RM008,Garlic,kg,586,2.04,4.39,3.122150,3.110,0.405220
8,RM009,Onion,kg,586,4.33,8.68,6.293106,6.290,0.804863
9,RM010,Spring Onion,kg,586,1.35,2.85,2.079676,2.070,0.276250


In [12]:
# Flag potential quantity outliers using the IQR method per raw material.
# This does NOT delete or modify any records.

outlier_results = []

for (material_id, material_name, unit), group in raw_df.groupby(
    ["material_id", "raw_material_name", "unit"]
):
    q1 = group["quantity_consumed"].quantile(0.25)
    q3 = group["quantity_consumed"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    flagged = group[
        (group["quantity_consumed"] < lower_bound) |
        (group["quantity_consumed"] > upper_bound)
    ]

    outlier_results.append({
        "material_id": material_id,
        "raw_material_name": material_name,
        "unit": unit,
        "outlier_count": len(flagged),
        "lower_bound": round(lower_bound, 3),
        "upper_bound": round(upper_bound, 3)
    })

outlier_report = pd.DataFrame(outlier_results)

print("Potential outlier review:")
print(outlier_report.to_string(index=False))

Potential outlier review:
material_id             raw_material_name unit  outlier_count  lower_bound  upper_bound
      RM001                        Chiton   kg              1       11.650       25.790
      RM002                          Salt   kg              3        3.425        7.065
      RM003                 Ground Pepper   kg              4        0.810        1.690
      RM004                Crushed Garlic   kg              1        1.655        3.535
      RM005 Optional Spices or Flavorings   kg              1        0.535        1.135
      RM006                   Cooking Oil    L              2        7.909       17.139
      RM007                  Small Shrimp   kg              3        4.805        9.725
      RM008                        Garlic   kg              1        2.029        4.219
      RM009                         Onion   kg              1        3.894        8.664
      RM010                  Spring Onion   kg              0        1.295        2.855
      

In [13]:
# Create a review table containing the actual potentially unusual records.
# Nothing is deleted or modified.

outlier_records = []

for (material_id, material_name, unit), group in raw_df.groupby(
    ["material_id", "raw_material_name", "unit"]
):
    q1 = group["quantity_consumed"].quantile(0.25)
    q3 = group["quantity_consumed"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    flagged = group[
        (group["quantity_consumed"] < lower_bound) |
        (group["quantity_consumed"] > upper_bound)
    ].copy()

    if not flagged.empty:
        flagged["outlier_lower_bound"] = lower_bound
        flagged["outlier_upper_bound"] = upper_bound
        flagged["review_status"] = "REVIEW_ONLY"

        outlier_records.append(flagged)

if outlier_records:
    outlier_review = pd.concat(
        outlier_records,
        ignore_index=True
    )
else:
    outlier_review = pd.DataFrame()

print("Potential records requiring review:",
      len(outlier_review))

outlier_review.head(20)

Potential records requiring review: 77


,material_id,date_consumed,raw_material_name,unit,quantity_consumed,outlier_lower_bound,outlier_upper_bound,review_status
0,RM001,2025-01-11,Chiton,kg,26.28,11.65000,25.79000,REVIEW_ONLY
1,RM002,2025-03-08,Salt,kg,7.32,3.42500,7.06500,REVIEW_ONLY
2,RM002,2025-08-25,Salt,kg,3.13,3.42500,7.06500,REVIEW_ONLY
3,RM002,2026-02-07,Salt,kg,7.76,3.42500,7.06500,REVIEW_ONLY
4,RM003,2025-05-03,Ground Pepper,kg,1.82,0.81000,1.69000,REVIEW_ONLY
5,RM003,2025-07-05,Ground Pepper,kg,1.76,0.81000,1.69000,REVIEW_ONLY
6,RM003,2025-10-04,Ground Pepper,kg,1.70,0.81000,1.69000,REVIEW_ONLY
7,RM003,2025-11-08,Ground Pepper,kg,1.75,0.81000,1.69000,REVIEW_ONLY
8,RM004,2026-06-06,Crushed Garlic,kg,3.80,1.65500,3.53500,REVIEW_ONLY
9,RM005,2026-01-19,Optional Spices or Flavorings,kg,0.51,0.53500,1.13500,REVIEW_ONLY


In [14]:
# Expected historical calendar period
expected_start = pd.Timestamp("2025-01-01")
expected_end = pd.Timestamp("2026-08-09")

expected_dates = pd.date_range(
    start=expected_start,
    end=expected_end,
    freq="D"
)

print("Expected calendar days:", len(expected_dates))
print("Expected start:", expected_start.date())
print("Expected end:", expected_end.date())

Expected calendar days: 586
Expected start: 2025-01-01
Expected end: 2026-08-09


In [15]:
# Check whether every material has records for every expected calendar day

coverage_results = []

for material_id, group in raw_df.groupby("material_id"):

    actual_dates = pd.DatetimeIndex(
        group["date_consumed"].dropna().unique()
    )

    missing_dates = expected_dates.difference(
        actual_dates
    )

    coverage_results.append({
        "material_id": material_id,
        "records": len(group),
        "expected_days": len(expected_dates),
        "missing_days": len(missing_dates),
        "first_date": group["date_consumed"].min(),
        "last_date": group["date_consumed"].max()
    })

coverage_report = pd.DataFrame(coverage_results)

print(coverage_report.to_string(index=False))

material_id  records  expected_days  missing_days first_date  last_date
      RM001      586            586             0 2025-01-01 2026-08-09
      RM002      586            586             0 2025-01-01 2026-08-09
      RM003      586            586             0 2025-01-01 2026-08-09
      RM004      586            586             0 2025-01-01 2026-08-09
      RM005      586            586             0 2025-01-01 2026-08-09
      RM006      586            586             0 2025-01-01 2026-08-09
      RM007      586            586             0 2025-01-01 2026-08-09
      RM008      586            586             0 2025-01-01 2026-08-09
      RM009      586            586             0 2025-01-01 2026-08-09
      RM010      586            586             0 2025-01-01 2026-08-09
      RM011      586            586             0 2025-01-01 2026-08-09
      RM012      586            586             0 2025-01-01 2026-08-09
      RM013      586            586             0 2025-01-01 202

In [16]:
# Final training-period boundary check

TRAINING_START = pd.Timestamp("2025-01-01")
TRAINING_END = pd.Timestamp("2026-08-09")

before_start = raw_df[
    raw_df["date_consumed"] < TRAINING_START
]

after_end = raw_df[
    raw_df["date_consumed"] > TRAINING_END
]

print("Records before training start:", len(before_start))
print("Records after training end:", len(after_end))

print("\nTraining start:", TRAINING_START.date())
print("Training end:", TRAINING_END.date())

if len(before_start) == 0 and len(after_end) == 0:
    print("\nTRAINING DATE GATE: PASSED")
else:
    print("\nTRAINING DATE GATE: FAILED")

Records before training start: 0
Records after training end: 0

Training start: 2025-01-01
Training end: 2026-08-09

TRAINING DATE GATE: PASSED


In [17]:
# Create the final validated training dataset.
# This does not invent, delete, or modify consumption values.

final_training_df = raw_df[
    [
        "material_id",
        "date_consumed",
        "raw_material_name",
        "unit",
        "quantity_consumed"
    ]
].copy()

# Keep the records in chronological order per raw material.
final_training_df = (
    final_training_df
    .sort_values(
        ["material_id", "date_consumed"]
    )
    .reset_index(drop=True)
)

print("=== FINAL TRAINING DATASET ===")
print("Records:", len(final_training_df))
print(
    "Materials:",
    final_training_df["material_id"].nunique()
)
print(
    "Start:",
    final_training_df["date_consumed"].min().date()
)
print(
    "End:",
    final_training_df["date_consumed"].max().date()
)

print("\nColumns:")
print(final_training_df.columns.tolist())

final_training_df.head()

=== FINAL TRAINING DATASET ===
Records: 17580
Materials: 30
Start: 2025-01-01
End: 2026-08-09

Columns:
['material_id', 'date_consumed', 'raw_material_name', 'unit', 'quantity_consumed']


,material_id,date_consumed,raw_material_name,unit,quantity_consumed
0,RM001,2025-01-01,Chiton,kg,16.91
1,RM001,2025-01-02,Chiton,kg,21.81
2,RM001,2025-01-03,Chiton,kg,21.11
3,RM001,2025-01-04,Chiton,kg,22.09
4,RM001,2025-01-05,Chiton,kg,22.22


In [18]:
# ============================================================
# RMIMS V2 — FINAL TRAINING DATA INTEGRITY GATE
# ============================================================

checks = {
    "required_columns_present": set([
        "material_id",
        "date_consumed",
        "raw_material_name",
        "unit",
        "quantity_consumed"
    ]).issubset(final_training_df.columns),

    "no_missing_material_id": (
        final_training_df["material_id"].isna().sum() == 0
    ),

    "no_missing_dates": (
        final_training_df["date_consumed"].isna().sum() == 0
    ),

    "no_missing_material_names": (
        final_training_df["raw_material_name"].isna().sum() == 0
    ),

    "no_missing_units": (
        final_training_df["unit"].isna().sum() == 0
    ),

    "no_missing_quantities": (
        final_training_df["quantity_consumed"].isna().sum() == 0
    ),

    "no_negative_quantities": (
        (final_training_df["quantity_consumed"] < 0).sum() == 0
    ),

    "no_zero_quantities": (
        (final_training_df["quantity_consumed"] == 0).sum() == 0
    ),

    "no_duplicate_rows": (
        final_training_df.duplicated().sum() == 0
    ),

    "exactly_30_materials": (
        final_training_df["material_id"].nunique() == 30
    ),

    "training_start_correct": (
        final_training_df["date_consumed"].min()
        == pd.Timestamp("2025-01-01")
    ),

    "training_end_correct": (
        final_training_df["date_consumed"].max()
        == pd.Timestamp("2026-08-09")
    ),

    "no_future_training_records": (
        (
            final_training_df["date_consumed"]
            > pd.Timestamp("2026-08-09")
        ).sum() == 0
    )
}

print("========================================")
print("RMIMS V2 — FINAL TRAINING DATA GATE")
print("========================================")

for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:5} | {check}")

print("========================================")

if all(checks.values()):
    print("FINAL TRAINING DATA GATE: PASSED")
else:
    print("FINAL TRAINING DATA GATE: FAILED")
    print("DO NOT PROCEED TO MODEL TRAINING.")

RMIMS V2 — FINAL TRAINING DATA GATE
PASS  | required_columns_present
PASS  | no_missing_material_id
PASS  | no_missing_dates
PASS  | no_missing_material_names
PASS  | no_missing_units
PASS  | no_missing_quantities
PASS  | no_negative_quantities
PASS  | no_zero_quantities
PASS  | no_duplicate_rows
PASS  | exactly_30_materials
PASS  | training_start_correct
PASS  | training_end_correct
PASS  | no_future_training_records
FINAL TRAINING DATA GATE: PASSED


In [19]:
# Save the validated training dataset
# This file contains ONLY the approved training period:
# 2025-01-01 through 2026-08-09.

processed_file = (
    Path.cwd()
    / "RMIMS_FINAL_TRAINING_DATA.csv"
)

final_training_df.to_csv(
    processed_file,
    index=False
)

print("Processed training dataset saved.")
print("File:", processed_file)
print("Rows:", len(final_training_df))
print(
    "Date range:",
    final_training_df["date_consumed"].min().date(),
    "to",
    final_training_df["date_consumed"].max().date()
)

Processed training dataset saved.
File: C:\Users\Zeanna\anaconda_projects\b33c9bb6-b656-4486-b51f-58f66e294016\RMIMS_FINAL_TRAINING_DATA.csv
Rows: 17580
Date range: 2025-01-01 to 2026-08-09


In [20]:
# ============================================================
# CELL 19 — PREPARE INDIVIDUAL RAW-MATERIAL TIME SERIES
# ============================================================

# Make sure dates are sorted chronologically
final_training_df = final_training_df.sort_values(
    ["material_id", "date_consumed"]
).reset_index(drop=True)

# Create a dictionary containing one time series per material.
material_series = {}

for material_id, group in final_training_df.groupby("material_id"):

    material_name = group["raw_material_name"].iloc[0]
    unit = group["unit"].iloc[0]

    series = (
        group
        .set_index("date_consumed")["quantity_consumed"]
        .sort_index()
    )

    material_series[material_id] = {
        "material_name": material_name,
        "unit": unit,
        "series": series
    }

print("Materials prepared:", len(material_series))

for material_id, information in list(material_series.items())[:5]:
    print(
        material_id,
        "|",
        information["material_name"],
        "|",
        information["unit"],
        "| observations:",
        len(information["series"])
    )

Materials prepared: 30
RM001 | Chiton | kg | observations: 586
RM002 | Salt | kg | observations: 586
RM003 | Ground Pepper | kg | observations: 586
RM004 | Crushed Garlic | kg | observations: 586
RM005 | Optional Spices or Flavorings | kg | observations: 586


In [21]:
# ============================================================
# CELL 20 — VERIFY ALL TIME SERIES
# ============================================================

series_check = []

for material_id, information in material_series.items():

    series = information["series"]

    series_check.append({
        "material_id": material_id,
        "raw_material_name": information["material_name"],
        "unit": information["unit"],
        "observations": len(series),
        "start_date": series.index.min(),
        "end_date": series.index.max(),
        "missing_values": series.isna().sum()
    })

series_check_df = pd.DataFrame(series_check)

print(series_check_df.to_string(index=False))

print("\n========================================")
print("TIME-SERIES VALIDATION")
print("========================================")

print(
    "Total materials:",
    len(series_check_df)
)

print(
    "Materials with 586 observations:",
    (
        series_check_df["observations"] == 586
    ).sum()
)

print(
    "Materials with missing values:",
    (
        series_check_df["missing_values"] > 0
    ).sum()
)

print(
    "Earliest date:",
    series_check_df["start_date"].min()
)

print(
    "Latest date:",
    series_check_df["end_date"].max()
)

material_id             raw_material_name unit  observations start_date   end_date  missing_values
      RM001                        Chiton   kg           586 2025-01-01 2026-08-09               0
      RM002                          Salt   kg           586 2025-01-01 2026-08-09               0
      RM003                 Ground Pepper   kg           586 2025-01-01 2026-08-09               0
      RM004                Crushed Garlic   kg           586 2025-01-01 2026-08-09               0
      RM005 Optional Spices or Flavorings   kg           586 2025-01-01 2026-08-09               0
      RM006                   Cooking Oil    L           586 2025-01-01 2026-08-09               0
      RM007                  Small Shrimp   kg           586 2025-01-01 2026-08-09               0
      RM008                        Garlic   kg           586 2025-01-01 2026-08-09               0
      RM009                         Onion   kg           586 2025-01-01 2026-08-09               0
      RM01

In [22]:
# ============================================================
# CELL 21 — HISTORICAL VALIDATION SPLIT
# ============================================================

VALIDATION_DAYS = 7

validation_end = final_training_df["date_consumed"].max()

validation_start = (
    validation_end
    - pd.Timedelta(days=VALIDATION_DAYS - 1)
)

model_training_df = final_training_df[
    final_training_df["date_consumed"] < validation_start
].copy()

historical_validation_df = final_training_df[
    final_training_df["date_consumed"] >= validation_start
].copy()

print("========================================")
print("HISTORICAL VALIDATION SPLIT")
print("========================================")

print(
    "Model training:",
    model_training_df["date_consumed"].min().date(),
    "to",
    model_training_df["date_consumed"].max().date()
)

print(
    "Historical validation:",
    historical_validation_df["date_consumed"].min().date(),
    "to",
    historical_validation_df["date_consumed"].max().date()
)

print(
    "Model training rows:",
    len(model_training_df)
)

print(
    "Validation rows:",
    len(historical_validation_df)
)

HISTORICAL VALIDATION SPLIT
Model training: 2025-01-01 to 2026-08-02
Historical validation: 2026-08-03 to 2026-08-09
Model training rows: 17370
Validation rows: 210


In [23]:
# ============================================================
# CELL 22 — PREPARE TRAINING AND VALIDATION SERIES
# ============================================================

model_series = {}
validation_series = {}

for material_id, group in model_training_df.groupby("material_id"):

    material_name = group["raw_material_name"].iloc[0]
    unit = group["unit"].iloc[0]

    train_series = (
        group
        .sort_values("date_consumed")
        .set_index("date_consumed")["quantity_consumed"]
    )

    validation_group = historical_validation_df[
        historical_validation_df["material_id"] == material_id
    ].sort_values("date_consumed")

    valid_series = (
        validation_group
        .set_index("date_consumed")["quantity_consumed"]
    )

    model_series[material_id] = {
        "material_name": material_name,
        "unit": unit,
        "series": train_series
    }

    validation_series[material_id] = {
        "material_name": material_name,
        "unit": unit,
        "series": valid_series
    }

print("Model-training series:", len(model_series))
print("Validation series:", len(validation_series))

print("\nFirst 5 materials:")

for material_id in list(model_series.keys())[:5]:

    print(
        material_id,
        "|",
        model_series[material_id]["material_name"],
        "| training observations:",
        len(model_series[material_id]["series"]),
        "| validation observations:",
        len(validation_series[material_id]["series"])
    )

Model-training series: 30
Validation series: 30

First 5 materials:
RM001 | Chiton | training observations: 579 | validation observations: 7
RM002 | Salt | training observations: 579 | validation observations: 7
RM003 | Ground Pepper | training observations: 579 | validation observations: 7
RM004 | Crushed Garlic | training observations: 579 | validation observations: 7
RM005 | Optional Spices or Flavorings | training observations: 579 | validation observations: 7


In [24]:
# ============================================================
# CELL 23 — AUTO-REG LAG VALIDATION
# ============================================================

from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Candidate lag values to evaluate.
# These are model configurations, not hardcoded consumption data.
candidate_lags = [7, 14, 21, 28]

lag_results = []

for material_id, information in model_series.items():

    train_series = information["series"]
    actual_series = validation_series[material_id]["series"]

    material_name = information["material_name"]

    for lag in candidate_lags:

        try:
            model = AutoReg(
                train_series,
                lags=lag,
                trend="ct",
                old_names=False
            )

            fitted_model = model.fit()

            forecast = fitted_model.predict(
                start=len(train_series),
                end=len(train_series) + len(actual_series) - 1,
                dynamic=False
            )

            forecast = pd.Series(
                forecast,
                index=actual_series.index
            )

            mae = mean_absolute_error(
                actual_series,
                forecast
            )

            rmse = np.sqrt(
                mean_squared_error(
                    actual_series,
                    forecast
                )
            )

            lag_results.append({
                "material_id": material_id,
                "raw_material_name": material_name,
                "lag": lag,
                "MAE": mae,
                "RMSE": rmse,
                "status": "PASS"
            })

        except Exception as e:

            lag_results.append({
                "material_id": material_id,
                "raw_material_name": material_name,
                "lag": lag,
                "MAE": np.nan,
                "RMSE": np.nan,
                "status": f"FAILED: {str(e)}"
            })

lag_results_df = pd.DataFrame(lag_results)

print("Lag validation completed.")
print(
    lag_results_df
    .head(20)
    .to_string(index=False)
)

C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\deterministic.py:308: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_index = self._extend_index(index, steps, forecast_index)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\deterministic.py:308: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is 

Lag validation completed.
material_id             raw_material_name  lag      MAE     RMSE status
      RM001                        Chiton    7 2.316565 2.754199   PASS
      RM001                        Chiton   14 2.030985 2.272797   PASS
      RM001                        Chiton   21 1.979311 2.237304   PASS
      RM001                        Chiton   28 1.972768 2.204487   PASS
      RM002                          Salt    7 0.423122 0.466743   PASS
      RM002                          Salt   14 0.362303 0.415942   PASS
      RM002                          Salt   21 0.281460 0.302815   PASS
      RM002                          Salt   28 0.307036 0.358190   PASS
      RM003                 Ground Pepper    7 0.073689 0.091501   PASS
      RM003                 Ground Pepper   14 0.074289 0.086580   PASS
      RM003                 Ground Pepper   21 0.064589 0.072953   PASS
      RM003                 Ground Pepper   28 0.050990 0.059995   PASS
      RM004                Crushed Gar

C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\deterministic.py:308: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_index = self._extend_index(index, steps, forecast_index)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\deterministic.py:308: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_index = self._extend_index(index, steps, forecast_index)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473:

In [25]:
# ============================================================
# CELL 24 — SELECT BEST AUTOREG LAG PER MATERIAL
# ============================================================

# Keep only successfully evaluated configurations
valid_lag_results = lag_results_df[
    lag_results_df["status"] == "PASS"
].copy()

# Rank by lowest RMSE first, then lowest MAE
best_lag_df = (
    valid_lag_results
    .sort_values(
        ["material_id", "RMSE", "MAE"]
    )
    .groupby("material_id", as_index=False)
    .first()
)

best_lag_df = best_lag_df[
    [
        "material_id",
        "raw_material_name",
        "lag",
        "MAE",
        "RMSE",
        "status"
    ]
]

print("========================================")
print("BEST AUTOREG CONFIGURATION PER MATERIAL")
print("========================================")

print(
    best_lag_df.to_string(index=False)
)

print("\n========================================")
print("VALIDATION SUMMARY")
print("========================================")

print(
    "Materials evaluated:",
    best_lag_df["material_id"].nunique()
)

print(
    "Configurations evaluated:",
    len(valid_lag_results)
)

print(
    "Average validation MAE:",
    round(best_lag_df["MAE"].mean(), 4)
)

print(
    "Average validation RMSE:",
    round(best_lag_df["RMSE"].mean(), 4)
)

BEST AUTOREG CONFIGURATION PER MATERIAL
material_id             raw_material_name  lag      MAE     RMSE status
      RM001                        Chiton   28 1.972768 2.204487   PASS
      RM002                          Salt   21 0.281460 0.302815   PASS
      RM003                 Ground Pepper   28 0.050990 0.059995   PASS
      RM004                Crushed Garlic    7 0.202599 0.273541   PASS
      RM005 Optional Spices or Flavorings   14 0.049105 0.063153   PASS
      RM006                   Cooking Oil   21 0.839378 1.065861   PASS
      RM007                  Small Shrimp   28 0.503389 0.609502   PASS
      RM008                        Garlic    7 0.163106 0.203599   PASS
      RM009                         Onion   21 0.453698 0.554656   PASS
      RM010                  Spring Onion   28 0.115720 0.151286   PASS
      RM011                       Cabbage    7 0.133769 0.155878   PASS
      RM012                       Carrots   14 0.409507 0.466270   PASS
      RM013             

In [26]:
# ============================================================
# CELL 25 — FINAL AUTOREG MODEL TRAINING
# ============================================================

from statsmodels.tsa.ar_model import AutoReg

final_models = {}
training_summary = []

for material_id, information in material_series.items():

    material_name = information["material_name"]
    unit = information["unit"]
    series = information["series"]

    # Get the lag selected automatically during validation
    selected_row = best_lag_df[
        best_lag_df["material_id"] == material_id
    ]

    if selected_row.empty:
        print(
            f"SKIPPED: {material_id} - "
            f"No validated lag available."
        )
        continue

    selected_lag = int(
        selected_row["lag"].iloc[0]
    )

    # Final model uses ALL historical observations
    # through 2026-08-09.
    model = AutoReg(
        series,
        lags=selected_lag,
        trend="ct",
        old_names=False
    )

    fitted_model = model.fit()

    final_models[material_id] = {
        "material_name": material_name,
        "unit": unit,
        "lag": selected_lag,
        "model": fitted_model,
        "training_start": series.index.min(),
        "training_end": series.index.max(),
        "training_observations": len(series)
    }

    training_summary.append({
        "material_id": material_id,
        "raw_material_name": material_name,
        "unit": unit,
        "selected_lag": selected_lag,
        "training_observations": len(series),
        "training_start": series.index.min(),
        "training_end": series.index.max(),
        "status": "TRAINED"
    })

training_summary_df = pd.DataFrame(
    training_summary
)

print("========================================")
print("FINAL AUTOREG TRAINING")
print("========================================")

print(
    "Models successfully trained:",
    len(final_models)
)

print(
    "Expected models:",
    final_training_df["material_id"].nunique()
)

print("\nTraining period:")
print(
    final_training_df["date_consumed"].min().date(),
    "to",
    final_training_df["date_consumed"].max().date()
)

print("\nModel summary:")
print(
    training_summary_df.to_string(index=False)
)

if len(final_models) == final_training_df["material_id"].nunique():
    print("\nFINAL MODEL TRAINING: PASSED")
else:
    print("\nFINAL MODEL TRAINING: FAILED")

C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  

FINAL AUTOREG TRAINING
Models successfully trained: 30
Expected models: 30

Training period:
2025-01-01 to 2026-08-09

Model summary:
material_id             raw_material_name unit  selected_lag  training_observations training_start training_end  status
      RM001                        Chiton   kg            28                    586     2025-01-01   2026-08-09 TRAINED
      RM002                          Salt   kg            21                    586     2025-01-01   2026-08-09 TRAINED
      RM003                 Ground Pepper   kg            28                    586     2025-01-01   2026-08-09 TRAINED
      RM004                Crushed Garlic   kg             7                    586     2025-01-01   2026-08-09 TRAINED
      RM005 Optional Spices or Flavorings   kg            14                    586     2025-01-01   2026-08-09 TRAINED
      RM006                   Cooking Oil    L            21                    586     2025-01-01   2026-08-09 TRAINED
      RM007               

C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  

In [27]:
# ============================================================
# CELL 26 — SAVE FINAL AUTOREG MODELS
# ============================================================

import joblib
from pathlib import Path

models_directory = Path.cwd() / "RMIMS_FINAL_MODELS"
models_directory.mkdir(
    parents=True,
    exist_ok=True
)

saved_models = []

for material_id, information in final_models.items():

    model_file = (
        models_directory
        / f"{material_id}_AutoReg.pkl"
    )

    joblib.dump(
        information["model"],
        model_file
    )

    saved_models.append({
        "material_id": material_id,
        "raw_material_name": information["material_name"],
        "unit": information["unit"],
        "selected_lag": information["lag"],
        "model_file": str(model_file)
    })

saved_models_df = pd.DataFrame(saved_models)

print("========================================")
print("FINAL MODEL SAVE")
print("========================================")

print(
    "Models saved:",
    len(saved_models_df)
)

print(
    "Model directory:",
    models_directory
)

print("\nSaved models:")
print(
    saved_models_df.to_string(index=False)
)

if len(saved_models_df) == 30:
    print("\nMODEL SAVE GATE: PASSED")
else:
    print("\nMODEL SAVE GATE: FAILED")

FINAL MODEL SAVE
Models saved: 30
Model directory: C:\Users\Zeanna\anaconda_projects\b33c9bb6-b656-4486-b51f-58f66e294016\RMIMS_FINAL_MODELS

Saved models:
material_id             raw_material_name unit  selected_lag                                                                                                  model_file
      RM001                        Chiton   kg            28 C:\Users\Zeanna\anaconda_projects\b33c9bb6-b656-4486-b51f-58f66e294016\RMIMS_FINAL_MODELS\RM001_AutoReg.pkl
      RM002                          Salt   kg            21 C:\Users\Zeanna\anaconda_projects\b33c9bb6-b656-4486-b51f-58f66e294016\RMIMS_FINAL_MODELS\RM002_AutoReg.pkl
      RM003                 Ground Pepper   kg            28 C:\Users\Zeanna\anaconda_projects\b33c9bb6-b656-4486-b51f-58f66e294016\RMIMS_FINAL_MODELS\RM003_AutoReg.pkl
      RM004                Crushed Garlic   kg             7 C:\Users\Zeanna\anaconda_projects\b33c9bb6-b656-4486-b51f-58f66e294016\RMIMS_FINAL_MODELS\RM004_AutoReg.pkl

In [28]:
# ============================================================
# CELL 27 — VERIFY SAVED AUTOREG MODELS
# ============================================================

import joblib
from pathlib import Path

loaded_models = {}
model_load_results = []

for _, row in saved_models_df.iterrows():

    material_id = row["material_id"]
    model_file = Path(row["model_file"])

    try:
        loaded_model = joblib.load(model_file)

        loaded_models[material_id] = loaded_model

        model_load_results.append({
            "material_id": material_id,
            "raw_material_name": row["raw_material_name"],
            "model_exists": model_file.exists(),
            "model_loaded": True,
            "status": "PASS"
        })

    except Exception as e:

        model_load_results.append({
            "material_id": material_id,
            "raw_material_name": row["raw_material_name"],
            "model_exists": model_file.exists(),
            "model_loaded": False,
            "status": f"FAIL: {str(e)}"
        })

model_load_results_df = pd.DataFrame(
    model_load_results
)

print("========================================")
print("SAVED MODEL LOAD VERIFICATION")
print("========================================")

print(
    model_load_results_df.to_string(index=False)
)

print("\nModels successfully loaded:",
      len(loaded_models))

print("Expected models:",
      len(saved_models_df))

if (
    len(loaded_models)
    == len(saved_models_df)
    and
    model_load_results_df["status"]
    .eq("PASS")
    .all()
):
    print("\nMODEL LOAD GATE: PASSED")
else:
    print("\nMODEL LOAD GATE: FAILED")

SAVED MODEL LOAD VERIFICATION
material_id             raw_material_name  model_exists  model_loaded status
      RM001                        Chiton          True          True   PASS
      RM002                          Salt          True          True   PASS
      RM003                 Ground Pepper          True          True   PASS
      RM004                Crushed Garlic          True          True   PASS
      RM005 Optional Spices or Flavorings          True          True   PASS
      RM006                   Cooking Oil          True          True   PASS
      RM007                  Small Shrimp          True          True   PASS
      RM008                        Garlic          True          True   PASS
      RM009                         Onion          True          True   PASS
      RM010                  Spring Onion          True          True   PASS
      RM011                       Cabbage          True          True   PASS
      RM012                       Carrots     

In [29]:
# ============================================================
# CELL 28 — FORECASTING ENGINE TEST
# ============================================================

from datetime import timedelta

def generate_forecast(
    model,
    last_date,
    forecast_days
):
    """
    Generate future predictions from a trained AutoReg model.

    The model itself produces the predicted consumption values.
    No consumption values are hardcoded.
    """

    forecast = model.predict(
        start=model.nobs,
        end=model.nobs + forecast_days - 1,
        dynamic=False
    )

    forecast_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=forecast_days,
        freq="D"
    )

    forecast_df = pd.DataFrame({
        "date": forecast_dates,
        "forecast_quantity": forecast.values
    })

    return forecast_df


# Use the historical training endpoint only.
forecast_test_last_date = pd.Timestamp("2026-08-09")

# Test with 31 future days.
# This is ONLY an engine test, not the actual Aug 10–17 evaluation.
TEST_FORECAST_DAYS = 31

test_forecasts = {}

for material_id, information in final_models.items():

    model = information["model"]

    forecast_df = generate_forecast(
        model=model,
        last_date=forecast_test_last_date,
        forecast_days=TEST_FORECAST_DAYS
    )

    forecast_df["material_id"] = material_id
    forecast_df["raw_material_name"] = information["material_name"]
    forecast_df["unit"] = information["unit"]

    test_forecasts[material_id] = forecast_df


print("========================================")
print("FORECASTING ENGINE TEST")
print("========================================")

print(
    "Materials forecasted:",
    len(test_forecasts)
)

print(
    "Forecast horizon:",
    TEST_FORECAST_DAYS,
    "days"
)

print(
    "Forecast start:",
    min(
        df["date"].min()
        for df in test_forecasts.values()
    ).date()
)

print(
    "Forecast end:",
    max(
        df["date"].max()
        for df in test_forecasts.values()
    ).date()
)

print("\nExample — RM001:")
print(
    test_forecasts["RM001"].head(10).to_string(index=False)
)

if len(test_forecasts) == 30:
    print("\nFORECASTING ENGINE TEST: PASSED")
else:
    print("\nFORECASTING ENGINE TEST: FAILED")

FORECASTING ENGINE TEST
Materials forecasted: 30
Forecast horizon: 31 days
Forecast start: 2026-08-10
Forecast end: 2026-09-09

Example — RM001:
      date  forecast_quantity material_id raw_material_name unit
2026-08-10          16.678136       RM001            Chiton   kg
2026-08-11          17.976442       RM001            Chiton   kg
2026-08-12          17.641328       RM001            Chiton   kg
2026-08-13          19.213944       RM001            Chiton   kg
2026-08-14          20.408213       RM001            Chiton   kg
2026-08-15          18.942170       RM001            Chiton   kg
2026-08-16          18.516415       RM001            Chiton   kg
2026-08-17          14.519083       RM001            Chiton   kg
2026-08-18          16.431787       RM001            Chiton   kg
2026-08-19          17.053840       RM001            Chiton   kg

FORECASTING ENGINE TEST: PASSED


C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\deterministic.py:308: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_index = self._extend_index(index, steps, forecast_index)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\deterministic.py:308: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_index = self._extend_index(index, steps, forecast_index)
C:\Users\Zeanna\anaconda3\Lib\site-packages\statsmodels\tsa\deterministic.py:308: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data lengt

In [30]:
# ============================================================
# CELL 29 — REBUILD FINAL MODELS WITH EXPLICIT DAILY FREQUENCY
# ============================================================

final_models = {}
training_summary = []

for material_id, information in material_series.items():

    material_name = information["material_name"]
    unit = information["unit"]
    series = information["series"].copy()

    # Explicitly establish the validated daily frequency.
    series.index = pd.DatetimeIndex(series.index)

    series = series.asfreq("D")

    # Get automatically selected lag
    selected_row = best_lag_df[
        best_lag_df["material_id"] == material_id
    ]

    if selected_row.empty:
        raise ValueError(
            f"No validated lag found for {material_id}"
        )

    selected_lag = int(
        selected_row["lag"].iloc[0]
    )

    model = AutoReg(
        series,
        lags=selected_lag,
        trend="ct",
        old_names=False
    )

    fitted_model = model.fit()

    final_models[material_id] = {
        "material_name": material_name,
        "unit": unit,
        "lag": selected_lag,
        "model": fitted_model,
        "training_start": series.index.min(),
        "training_end": series.index.max(),
        "training_observations": len(series)
    }

    training_summary.append({
        "material_id": material_id,
        "raw_material_name": material_name,
        "unit": unit,
        "selected_lag": selected_lag,
        "training_observations": len(series),
        "training_start": series.index.min(),
        "training_end": series.index.max(),
        "status": "TRAINED"
    })

training_summary_df = pd.DataFrame(
    training_summary
)

print("========================================")
print("FINAL AUTOREG MODELS — DAILY FREQUENCY")
print("========================================")

print(
    "Models successfully trained:",
    len(final_models)
)

print(
    "Expected models:",
    final_training_df["material_id"].nunique()
)

print(
    "Training start:",
    final_training_df["date_consumed"].min().date()
)

print(
    "Training end:",
    final_training_df["date_consumed"].max().date()
)

if len(final_models) == 30:
    print("\nFINAL MODEL REBUILD: PASSED")
else:
    print("\nFINAL MODEL REBUILD: FAILED")
    

FINAL AUTOREG MODELS — DAILY FREQUENCY
Models successfully trained: 30
Expected models: 30
Training start: 2025-01-01
Training end: 2026-08-09

FINAL MODEL REBUILD: PASSED


In [31]:
# ============================================================
# CELL 30 — SAVE REBUILT FINAL AUTOREG MODELS
# ============================================================

models_directory = Path.cwd() / "RMIMS_FINAL_MODELS"
models_directory.mkdir(
    parents=True,
    exist_ok=True
)

rebuilt_model_records = []

for material_id, information in final_models.items():

    model_file = (
        models_directory
        / f"{material_id}_AutoReg.pkl"
    )

    joblib.dump(
        information["model"],
        model_file
    )

    rebuilt_model_records.append({
        "material_id": material_id,
        "raw_material_name": information["material_name"],
        "unit": information["unit"],
        "selected_lag": information["lag"],
        "training_start": information["training_start"],
        "training_end": information["training_end"],
        "training_observations": information["training_observations"],
        "model_file": str(model_file)
    })

rebuilt_models_df = pd.DataFrame(
    rebuilt_model_records
)

print("========================================")
print("REBUILT MODEL SAVE")
print("========================================")

print(
    "Models saved:",
    len(rebuilt_models_df)
)

print(
    "Training period:",
    rebuilt_models_df["training_start"].min().date(),
    "to",
    rebuilt_models_df["training_end"].max().date()
)

if len(rebuilt_models_df) == 30:
    print("\nREBUILT MODEL SAVE GATE: PASSED")
else:
    print("\nREBUILT MODEL SAVE GATE: FAILED")

REBUILT MODEL SAVE
Models saved: 30
Training period: 2025-01-01 to 2026-08-09

REBUILT MODEL SAVE GATE: PASSED


In [32]:
# ============================================================
# CELL 31 — WEEKLY AND MONTHLY FORECAST AGGREGATION
# ============================================================

def build_forecast_support(
    models,
    forecast_start,
    forecast_days=31
):
    """
    Generate model predictions and aggregate them into:
    - weekly forecast support
    - monthly forecast support

    Forecast quantities are generated dynamically by AutoReg.
    No forecast quantities are hardcoded.
    """

    all_forecasts = []

    forecast_start = pd.Timestamp(forecast_start)

    for material_id, information in models.items():

        model = information["model"]

        forecast = model.predict(
            start=model.nobs,
            end=model.nobs + forecast_days - 1,
            dynamic=False
        )

        dates = pd.date_range(
            start=forecast_start,
            periods=forecast_days,
            freq="D"
        )

        material_forecast = pd.DataFrame({
            "material_id": material_id,
            "raw_material_name": information["material_name"],
            "unit": information["unit"],
            "date": dates,
            "forecast_quantity": forecast.to_numpy()
        })

        all_forecasts.append(material_forecast)

    daily_forecast_df = pd.concat(
        all_forecasts,
        ignore_index=True
    )

    # Prevent negative model predictions from becoming
    # negative physical requirements.
    daily_forecast_df["forecast_quantity"] = (
        daily_forecast_df["forecast_quantity"]
        .clip(lower=0)
    )

    # --------------------------------------------------------
    # WEEKLY FORECAST
    # --------------------------------------------------------

    daily_forecast_df["week_start"] = (
        daily_forecast_df["date"]
        - pd.to_timedelta(
            daily_forecast_df["date"].dt.weekday,
            unit="D"
        )
    )

    weekly_forecast_df = (
        daily_forecast_df
        .groupby(
            [
                "material_id",
                "raw_material_name",
                "unit",
                "week_start"
            ],
            as_index=False
        )["forecast_quantity"]
        .sum()
        .rename(
            columns={
                "forecast_quantity":
                "weekly_forecast_quantity"
            }
        )
    )

    # --------------------------------------------------------
    # MONTHLY FORECAST
    # --------------------------------------------------------

    daily_forecast_df["month"] = (
        daily_forecast_df["date"]
        .dt.to_period("M")
    )

    monthly_forecast_df = (
        daily_forecast_df
        .groupby(
            [
                "material_id",
                "raw_material_name",
                "unit",
                "month"
            ],
            as_index=False
        )["forecast_quantity"]
        .sum()
        .rename(
            columns={
                "forecast_quantity":
                "monthly_forecast_quantity"
            }
        )
    )

    return (
        daily_forecast_df,
        weekly_forecast_df,
        monthly_forecast_df
    )


# ------------------------------------------------------------
# TEST THE SUPPORT ENGINE
# ------------------------------------------------------------

(
    forecast_engine_daily,
    weekly_forecast_test,
    monthly_forecast_test
) = build_forecast_support(
    models=final_models,
    forecast_start=pd.Timestamp("2026-08-10"),
    forecast_days=31
)

print("========================================")
print("FORECAST SUPPORT ENGINE")
print("========================================")

print(
    "Materials:",
    weekly_forecast_test["material_id"].nunique()
)

print(
    "Weekly forecast rows:",
    len(weekly_forecast_test)
)

print(
    "Monthly forecast rows:",
    len(monthly_forecast_test)
)

print("\nWEEKLY FORECAST SAMPLE:")
print(
    weekly_forecast_test
    .head(10)
    .to_string(index=False)
)

print("\nMONTHLY FORECAST SAMPLE:")
print(
    monthly_forecast_test
    .head(10)
    .to_string(index=False)
)

print("\nFORECAST SUPPORT ENGINE: PASSED")

FORECAST SUPPORT ENGINE
Materials: 30
Weekly forecast rows: 150
Monthly forecast rows: 60

WEEKLY FORECAST SAMPLE:
material_id raw_material_name unit week_start  weekly_forecast_quantity
      RM001            Chiton   kg 2026-08-10                129.376649
      RM001            Chiton   kg 2026-08-17                123.413116
      RM001            Chiton   kg 2026-08-24                128.499232
      RM001            Chiton   kg 2026-08-31                137.932505
      RM001            Chiton   kg 2026-09-07                 53.601823
      RM002              Salt   kg 2026-08-10                 35.965013
      RM002              Salt   kg 2026-08-17                 35.055640
      RM002              Salt   kg 2026-08-24                 37.444346
      RM002              Salt   kg 2026-08-31                 37.678402
      RM002              Salt   kg 2026-09-07                 13.983131

MONTHLY FORECAST SAMPLE:
material_id             raw_material_name unit   month  monthly_for

In [33]:
# ============================================================
# CELL 32 — VERIFY WEEKLY/MONTHLY AGGREGATION
# ============================================================

# Verify that weekly totals equal the sum of their
# underlying forecast values.

weekly_check = (
    forecast_engine_daily
    .groupby(
        [
            "material_id",
            "raw_material_name",
            "unit",
            "week_start"
        ],
        as_index=False
    )["forecast_quantity"]
    .sum()
    .rename(
        columns={
            "forecast_quantity":
            "calculated_weekly_quantity"
        }
    )
)

weekly_comparison = weekly_forecast_test.merge(
    weekly_check,
    on=[
        "material_id",
        "raw_material_name",
        "unit",
        "week_start"
    ],
    how="left"
)

weekly_comparison["difference"] = (
    weekly_comparison["weekly_forecast_quantity"]
    -
    weekly_comparison["calculated_weekly_quantity"]
)

# Verify monthly totals
monthly_check = (
    forecast_engine_daily
    .groupby(
        [
            "material_id",
            "raw_material_name",
            "unit",
            "month"
        ],
        as_index=False
    )["forecast_quantity"]
    .sum()
    .rename(
        columns={
            "forecast_quantity":
            "calculated_monthly_quantity"
        }
    )
)

monthly_comparison = monthly_forecast_test.merge(
    monthly_check,
    on=[
        "material_id",
        "raw_material_name",
        "unit",
        "month"
    ],
    how="left"
)

monthly_comparison["difference"] = (
    monthly_comparison["monthly_forecast_quantity"]
    -
    monthly_comparison["calculated_monthly_quantity"]
)

print("========================================")
print("FORECAST AGGREGATION VALIDATION")
print("========================================")

print(
    "Maximum weekly calculation difference:",
    weekly_comparison["difference"].abs().max()
)

print(
    "Maximum monthly calculation difference:",
    monthly_comparison["difference"].abs().max()
)

if (
    weekly_comparison["difference"].abs().max() < 1e-10
    and
    monthly_comparison["difference"].abs().max() < 1e-10
):
    print("\nWEEKLY/MONTHLY AGGREGATION: PASSED")
else:
    print("\nWEEKLY/MONTHLY AGGREGATION: FAILED")

FORECAST AGGREGATION VALIDATION
Maximum weekly calculation difference: 0.0
Maximum monthly calculation difference: 0.0

WEEKLY/MONTHLY AGGREGATION: PASSED
